<a href="https://colab.research.google.com/github/Hamid-Ghalandari/XAI/blob/main/An_XAI_for_public_health_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Splitting into train and test

In [ ]:

# @title Stratiffied splitting (based on gender)

from sklearn.model_selection import train_test_split

strat_train_set, strat_test_set = train_test_split(
    df_final, test_size = 0.2, stratify = df_final['RIAGENDR'], random_state=42)


# To verify the function of the splitter (How similar the proportions are regarding to the variables based on which the dataframe was stratified)
print(strat_test_set['RIAGENDR'].value_counts()/len(strat_test_set))
print(df_final['RIAGENDR'].value_counts()/len(df_final))



In [ ]:
# @title Creating a copy of the train set

df_obesity = strat_train_set.copy()
# df_obesity
# print(df_obesity.columns)

# Preparing the dataset for machine learning

In [ ]:
# @title Imputation of missing values

df_obesity_cat = df_obesity.select_dtypes(include = ['category'])
df_obesity_num = df_obesity.select_dtypes(include = [np.number])

from sklearn.impute import SimpleImputer

imputer_cat = SimpleImputer(strategy= 'most_frequent')
imputer_num = SimpleImputer(strategy= 'median')


print(imputer_cat.fit(df_obesity_cat))
print(imputer_num.fit(df_obesity_num))

#%% Testing

print(imputer_cat.statistics_)
print(df_obesity_cat.mode().values)

print(imputer_num.statistics_)
print(df_obesity_num.median().values)



In [ ]:
# @title One-hot encoding of categorical variables

#**Should be reconsidered if the final model is underfit!**

from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder(handle_unknown= 'ignore')
df_obesity_cat_1hot = cat_encoder.fit_transform(df_obesity_cat)

df_obesity_cat_1hot
# df_obesity_cat_1hot.toarray()

In [ ]:
# @title Scaling and transformation of numerical data

# Trnasformation of input features

#1: Yeo-Johnson method (chosen over 'log' method, because data may contain zero values)

from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

transformer_num_feat = PowerTransformer(method = 'yeo-johnson')
obesity_num_trf = transformer_num_feat.fit_transform(df_obesity_num)
obesity_num_trf

scaler_num_feat = RobustScaler()
obesity_num_trf_scaled = scaler_num_feat.fit_transform(obesity_num_trf)

obesity_num_trf_scaled








In [ ]:

# @title Visualizing the impact of transformation

# import matplotlib.pyplot as plt
# import seaborn as sns
# from scipy import stats

# # %%capture

# # Histograms before and after transformation
# for column in df_obesity_num.columns:
#     sns.histplot(df_obesity_num[column], kde=True, label='Original')
#     sns.histplot(obesity_num_trf_scaled[:, df_obesity_num.columns.tolist().index(column)], kde=True, label='Transformed and Scaled')
#     plt.title(column)
#     plt.legend()
#     plt.show()

# # Q-Q plots before and after transformation
# for column in df_obesity_num.columns:
#     plt.figure(figsize=(12, 6))

#     plt.subplot(1, 2, 1)
#     stats.probplot(df_obesity_num[column], dist="norm", plot=plt)
#     plt.title('Original ' + column)

#     plt.subplot(1, 2, 2)
#     stats.probplot(obesity_num_trf_scaled[:, df_obesity_num.columns.tolist().index(column)], dist="norm", plot=plt)
#     plt.title('Transformed and Scaled ' + column)

#     plt.show()


In [ ]:
# @title Visualization following scaling and transformation

# %%capture
# Input features

# new_columns = df_obesity_num.columns

# sample_transf_scaled_df = pd.DataFrame(obesity_num_trf_scaled, columns= new_columns, index= df_obesity_num.index)
# sample_transf_scaled_df

# from pandas.plotting import scatter_matrix

# some_num_attributes_input = ['RIDAGEYR', 'INDFMPIR', 'Calorie_intake_mean', 'Protein_intake_mean', 'Carb_intake_mean',
#        'Cholesterol_intake_mean', 'INDFMMPI', 'PAD680', 'SLD012', 'SLD013','HEI-2020']

# scatter_matrix(sample_transf_scaled_df[some_num_attributes_input], figsize = (40,30))
# plt.show()

# Transformation Pipelines

In [ ]:
# @title Numerical variables

from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
   ('impute', SimpleImputer(strategy= 'median')),
   ('transform', PowerTransformer(method = 'yeo-johnson')),
   ('standardize', RobustScaler())
])
num_pipeline

Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                ('transform', PowerTransformer()),
                ('standardize', RobustScaler())])

In [ ]:
# @title Applying 'num_pipeline' to numerical attributes

obesity_num_prepared = num_pipeline.fit_transform(df_obesity_num)
obesity_num_prepared


# Creating a dataframe
df_obesity_num_prepared = pd.DataFrame(
    obesity_num_prepared, columns = num_pipeline.get_feature_names_out(),
    index = df_obesity_cat.index)
# df_obesity_num_prepared

In [ ]:
# @title Categorical variables

from sklearn.pipeline import Pipeline

cat_pipeline = Pipeline([
   ('impute', SimpleImputer(strategy= 'most_frequent')),
   ('encoder', OneHotEncoder(handle_unknown= 'ignore')),
])
cat_pipeline

In [ ]:
# @title Applying 'cat_pipeline' to categorical attributes

from sklearn.compose import ColumnTransformer

obesity_cat_prepared = cat_pipeline.fit_transform(df_obesity_cat)
obesity_cat_prepared.shape


# # Getting the original column names
# original_feature_names = df_obesity_cat.columns

# encoded_feature_names = cat_pipeline.named_steps['encoder'].get_feature_names_out()

# new_feature_names = [original_feature_names[int(name.split('_')[0][1:])] + '_' + name.split('_')[1] for name in encoded_feature_names]


# # # Creating a dataframe
# df_obesity_cat_prepared = pd.DataFrame(
#     obesity_cat_prepared.toarray(), columns = new_feature_names,
#     index = df_obesity_num.index)
# df_obesity_cat_prepared.head(2)

In [ ]:
# @title Making a uniform transformer

from sklearn.compose import make_column_selector, make_column_transformer

preprocessing = make_column_transformer(
    (num_pipeline, make_column_selector(dtype_include=np.number)),
    (cat_pipeline, make_column_selector(dtype_include='category'))
)


df_obesity_prepared = preprocessing.fit_transform(df_obesity)


# df_obesity_prepared.shape

# preprocessing.get_feature_names_out()


In [ ]:
# @title Visualizing the preprocessing procedure

preprocessing

# Training the models

In [ ]:
# @title Decision tree classifier

sample_weights = weights_2_day
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline

# tree_class = make_pipeline (preprocessing, DecisionTreeClassifier(random_state=42, max_depth = 11, max_features = 73, min_samples_split = 69))
# tree_class.fit(df_obesity,df_obesity_labels_2, decisiontreeclassifier__sample_weight= sample_weights)


In [ ]:
# @title Testing the decision tree classifier

# obesity_prediction_tree_class = tree_class.predict(df_obesity)
# print(obesity_prediction_tree_class[:5].round(0))
# print((df_obesity_labels_2[:5]))

# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# tree_class_acc = accuracy_score(df_obesity_labels_2,obesity_prediction_tree_class,
#                                     sample_weight=sample_weights)

# tree_class_acc

In [ ]:
# @title Random forest classifier

sample_weights = weights_2_day
from sklearn.pipeline import make_pipeline

from sklearn.ensemble import RandomForestClassifier

# Hyperparameters where obtained by the randomized grid search below

forest_class = make_pipeline (preprocessing, RandomForestClassifier(random_state=42, max_depth=61, max_features=24, min_samples_split=81))
forest_class.fit(df_obesity,df_obesity_labels_2, randomforestclassifier__sample_weight= sample_weights)

In [ ]:
# @title Testing Random forest classifier

obesity_predict_frst_class = forest_class.predict(df_obesity)
print(obesity_predict_frst_class[:5].round(0))
print((df_obesity_labels_2[:5]))

from sklearn.metrics import mean_squared_error

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

forest_class_acc = accuracy_score(df_obesity_labels_2, obesity_predict_frst_class,
                                      sample_weight=sample_weights)
forest_class_acc

In [ ]:
# @title Support Vector Machine (classifier)

# from sklearn.svm import SVC

# SVC_class = make_pipeline (preprocessing, SVC(random_state= 42,  C=0.001, probability = True))
# SVC_class.fit(df_obesity,df_obesity_labels_2, svc__sample_weight= sample_weights)

In [ ]:
# @title Testing Support Vector Machine (classifier)


# obesity_predict_svc_class = SVC_class.predict(df_obesity)
# print(obesity_predict_frst_class[:5].round(0))
# print((df_obesity_labels_2[:5]))


# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# SVC_class_acc = accuracy_score(df_obesity_labels_2, obesity_predict_svc_class,
#                                       sample_weight=sample_weights)
# SVC_class_acc

In [ ]:
# @title KNN (classifier)

# from sklearn.neighbors import KNeighborsClassifier

# KNN_class = make_pipeline (preprocessing, KNeighborsClassifier(weights = 'distance', n_neighbors = 29))
# KNN_class.fit(df_obesity,df_obesity_labels_2)

In [ ]:
# @title Testing KNN (classifier)

# obesity_predict_knn_class = KNN_class.predict(df_obesity)
# print(obesity_predict_knn_class[:5].round(0))
# print((df_obesity_labels_2[:5]))



# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# KNN_class_acc = accuracy_score(df_obesity_labels_2, obesity_predict_knn_class,
#                                       sample_weight=sample_weights)
# KNN_class_acc

In [ ]:
# @title Gradient Boosting classifier

# !pip install xgboost

# sample_weights = weights_2_day

# from xgboost import XGBClassifier
# from sklearn.pipeline import make_pipeline




# XGB_class = make_pipeline (preprocessing, XGBClassifier(random_state=42))
# XGB_class.fit(df_obesity,df_obesity_labels_2, xgbclassifier__sample_weight= sample_weights)


In [ ]:
# @title Testing Gradient Boosting

# obesity_predict_xgb_class = XGB_class.predict(df_obesity)
# print(obesity_predict_xgb_class[:5].round(0))
# print((df_obesity_labels_2[:5]))

# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# xgb_class_acc = accuracy_score(df_obesity_labels_2, obesity_predict_xgb_class,
#                                       sample_weight=sample_weights)
# xgb_class_acc

# Cross-validation of the models

In [ ]:
# @title Cross-validation of DT classifier


from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define the metrics
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Perform cross-validation with multiple metrics
tree_class_scores = cross_validate(tree_class, df_obesity, df_obesity_labels_2,
                                   scoring=scoring, cv=10,
                                   fit_params={'decisiontreeclassifier__sample_weight': sample_weights})



# Extract the scores for each metric
accuracy_scores_DT = tree_class_scores['test_accuracy']
print(pd.Series(accuracy_scores_DT).describe())
precision_scores_DT = tree_class_scores['test_precision']
print(pd.Series(precision_scores_DT).describe())
recall_scores_DT = tree_class_scores['test_recall']
print(pd.Series(recall_scores_DT).describe())
f1_scores_DT = tree_class_scores['test_f1']
print(pd.Series(f1_scores_DT).describe())
roc_auc_scores_DT = tree_class_scores['test_roc_auc']
print(pd.Series(roc_auc_scores_DT).describe())



In [ ]:
# @title Cross-validation of RF classifier

from sklearn.model_selection import cross_val_score

forest_class_acc_scores = cross_val_score(forest_class, df_obesity,df_obesity_labels_2,
                                         scoring = 'accuracy', cv =10,
                                           fit_params={'randomforestclassifier__sample_weight': sample_weights})

pd.Series(forest_class_acc_scores).describe()

from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define the metrics
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Perform cross-validation with multiple metrics
forest_class_scores = cross_validate(forest_class, df_obesity, df_obesity_labels_2,
                                   scoring=scoring, cv=10,
                                   fit_params={'randomforestclassifier__sample_weight': sample_weights})

# Extract the scores for each metric
accuracy_scores_RF = forest_class_scores['test_accuracy']
print(pd.Series(accuracy_scores_RF).describe())
precision_scores_RF = forest_class_scores['test_precision']
print(pd.Series(precision_scores_RF).describe())
recall_scores_RF = forest_class_scores['test_recall']
print(pd.Series(recall_scores_RF).describe())
f1_scores_RF = forest_class_scores['test_f1']
print(pd.Series(f1_scores_RF).describe())
roc_auc_scores_RF = forest_class_scores['test_roc_auc']
print(pd.Series(roc_auc_scores_RF).describe())

In [ ]:
# @title Cross-validation of SVC classifier


from sklearn.model_selection import cross_val_score

SVC_class_acc_score = cross_val_score(SVC_class, df_obesity,df_obesity_labels_2,
                                        scoring = 'accuracy', cv =10,
                                        fit_params={'svc__sample_weight': sample_weights})

pd.Series(SVC_class_acc_score).describe()

from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define the metrics
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Perform cross-validation with multiple metrics
SVC_class_scores = cross_validate(SVC_class, df_obesity, df_obesity_labels_2,
                                   scoring=scoring, cv=10,
                                   fit_params={'svc__sample_weight': sample_weights})

# Extract the scores for each metric
accuracy_scores_SVC = SVC_class_scores['test_accuracy']
print(pd.Series(accuracy_scores_SVC).describe())
precision_scores_SVC = SVC_class_scores['test_precision']
print(pd.Series(precision_scores_SVC).describe())
recall_scores_SVC = SVC_class_scores['test_recall']
print(pd.Series(recall_scores_SVC).describe())
f1_scores_SVC = SVC_class_scores['test_f1']
print(pd.Series(f1_scores_SVC).describe())
roc_auc_scores_SVC = SVC_class_scores['test_roc_auc']
print(pd.Series(roc_auc_scores_SVC).describe())

In [ ]:
# @title Cross-validation of KNN classifier


from sklearn.model_selection import cross_val_score

KNN_class_acc_score = cross_val_score(KNN_class, df_obesity,df_obesity_labels_2,
                                        scoring = 'accuracy', cv =10,
                                        )


pd.Series(KNN_class_acc_score).describe()

from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define the metrics
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Perform cross-validation with multiple metrics
KNN_class_scores = cross_validate(KNN_class, df_obesity, df_obesity_labels_2,
                                   scoring=scoring, cv=10,
                                   )

# Extract the scores for each metric
accuracy_scores_KNN = KNN_class_scores['test_accuracy']
print(pd.Series(accuracy_scores_KNN).describe())
precision_scores_KNN = KNN_class_scores['test_precision']
print(pd.Series(precision_scores_KNN).describe())
recall_scores_KNN = KNN_class_scores['test_recall']
print(pd.Series(recall_scores_KNN).describe())
f1_scores_KNN = KNN_class_scores['test_f1']
print(pd.Series(f1_scores_KNN).describe())
roc_auc_scores_KNN = KNN_class_scores['test_roc_auc']
print(pd.Series(roc_auc_scores_KNN).describe())

In [ ]:
# @title Cross validation of XGBoost Classifier



from sklearn.model_selection import cross_val_score

XGB_class_acc_score = cross_val_score(XGB_class, df_obesity,df_obesity_labels_2,
                                        scoring = 'accuracy', cv =10,
                                        fit_params={'xgbclassifier__sample_weight': sample_weights})

pd.Series(XGB_class_acc_score).describe()

from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define the metrics
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Perform cross-validation with multiple metrics
XGB_class_scores = cross_validate(XGB_class, df_obesity, df_obesity_labels_2,
                                   scoring=scoring, cv=10,
                                   fit_params={'xgbclassifier__sample_weight': sample_weights})

# Extract the scores for each metric
accuracy_scores_XGB = XGB_class_scores['test_accuracy']
print(pd.Series(accuracy_scores_XGB).describe())
precision_scores_XGB = XGB_class_scores['test_precision']
print(pd.Series(precision_scores_XGB).describe())
recall_scores_XGB = XGB_class_scores['test_recall']
print(pd.Series(recall_scores_XGB).describe())
f1_scores_XGB = XGB_class_scores['test_f1']
print(pd.Series(f1_scores_XGB).describe())
roc_auc_scores_XGB = XGB_class_scores['test_roc_auc']
print(pd.Series(roc_auc_scores_XGB).describe())

# Grid Search

In [ ]:
# @title Decision Tree Classifier (Hyperparameter Tuning) (Multiple parameters)

# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, f1_score
# from sklearn.tree import DecisionTreeClassifier



# # Create a pipeline
# full_pipeline_DT = Pipeline([
#     ('preprocessing', preprocessing),
#     ('tree_class', DecisionTreeClassifier(random_state=42))
# ])

# # Update the parameter distribution dictionary with multiple parameters
# param_distrbs = {
#     'tree_class__max_features': randint(low=10, high=100),
#     'tree_class__max_depth': randint(low=10, high=100),
#     'tree_class__min_samples_split': randint(low=10, high=100),
# }

# # Define multiple scoring metrics
# scoring = {
#     'accuracy': 'accuracy',
#     'precision': 'precision_macro',
#     'roc_auc': 'roc_auc_ovr',
#     'f1': 'f1_macro'
# }

# # Create RandomizedSearchCV with multiple scoring
# rnd_search_DT = RandomizedSearchCV(
#     full_pipeline_DT, param_distributions=param_distrbs, n_iter=10, cv=10,
#     scoring=scoring, refit='roc_auc', random_state=42, n_jobs=-1
# )

# # Fit the RandomizedSearchCV
# rnd_search_DT.fit(df_obesity, df_obesity_labels_2)

# # Get the best parameters for each scoring metric
# best_params_DT = rnd_search_DT.best_params_
# print("Best Parameters:")
# print(best_params_DT)

# # Get the best scores for each scoring metric
# best_scores_DT = rnd_search_DT.best_score_
# print("Best Scores:")
# print(best_scores_DT)


In [ ]:
# @title Random Forest Classifier (Hyperparameter Tuning) (Multiple parameters)

# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, f1_score
# from sklearn.ensemble import RandomForestClassifier


# # Create a pipeline
# full_pipeline = Pipeline([
#     ('preprocessing', preprocessing),
#     ('forest_class', RandomForestClassifier(random_state=42))
# ])

# # Update the parameter distribution dictionary with multiple parameters
# param_distrbs = {
#     'forest_class__max_features': randint(low=10, high=100),
#     'forest_class__max_depth': randint(low=10, high=100),
#     'forest_class__min_samples_split': randint(low=10, high=100),
# }

# # Define multiple scoring metrics
# scoring = {
#     'accuracy': 'accuracy',
#     'precision': 'precision_macro',
#     'roc_auc': 'roc_auc_ovr',
#     'f1': 'f1_macro'
# }

# # Create RandomizedSearchCV with multiple scoring
# rnd_search = RandomizedSearchCV(
#     full_pipeline, param_distributions=param_distrbs, n_iter=10, cv=10,
#     scoring=scoring, refit='roc_auc', random_state=42, n_jobs=-1
# )

# # Fit the RandomizedSearchCV
# rnd_search.fit(df_obesity, df_obesity_labels_2)

# # Get the best parameters for each scoring metric
# best_params = rnd_search.best_params_
# print("Best Parameters:")
# print(best_params)

# # Get the best scores for each scoring metric
# best_scores = rnd_search.best_score_
# print("Best Scores:")
# print(best_scores)


In [ ]:
# @title SVC Classifier (Hyperparameter Tuning)

# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.svm import SVC
# import numpy as np


# # Create a pipeline
# full_pipeline_svc = Pipeline([
#     ('preprocessing', preprocessing),
#     ('SVC_class', SVC(random_state=42))
# ])

# # Define the parameter values that should be searched
# param_dist = {'SVC_class__C': np.logspace(-3, 2, 6)}
#               # 'SVC_class__gamma': np.logspace(-3, 2, 6),
#               # 'SVC_class__kernel': ['linear', 'poly', 'rbf', 'sigmoid']}


# scoring = {
#     'accuracy': 'accuracy',
#     'precision': 'precision_macro',
#     'roc_auc': 'roc_auc_ovr',
#     'f1': 'f1_macro'
# }

# random_search_svc = RandomizedSearchCV(full_pipeline_svc, param_distributions=param_dist, n_iter=100, cv=5,
#                                    scoring = scoring, refit = 'roc_auc', verbose=2, random_state=42, n_jobs=-1)

# # Fit the model
# random_search_svc.fit(df_obesity, df_obesity_labels_2)

# # # Get the best parameters for each scoring metric
# best_params_svc = random_search_svc.best_params_
# print("Best Parameters:")
# print(best_params_svc)

# # Get the best scores for each scoring metric
# best_scores_svc = random_search_svc.best_score_
# print("Best Scores:")
# print(best_scores_svc)


In [ ]:
# @title XGB Classifier (Hyperparameter Tuning)


# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint
# from scipy.stats import uniform
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, f1_score



# # # Create a pipeline
# full_pipeline_xgb = Pipeline([
#     ('preprocessing', preprocessing),
#     ('XGB_class', XGBClassifier(random_state=42))
# ])

# param_distrbs = {
#     'XGB_class__n_estimators': randint(low=10, high=100),
#     'XGB_class__max_depth': randint(low=10, high=100),
#     'XGB_class__min_child_weight': randint(low=1, high=100),
#     'XGB_class__gamma': randint(low=0, high=100),
#     'XGB_class__subsample': uniform(loc=0.1, scale=0.9),
#     'XGB_class__colsample_bytree': uniform(loc=0.1, scale=0.9),
#     'XGB_class__learning_rate': [0.01, 0.02, 0.05, 0.1, 0.15, 0.2]
# }


# scoring = {
#     'accuracy': 'accuracy',
#     'precision': 'precision_macro',
#     'roc_auc': 'roc_auc_ovr',
#     'f1': 'f1_macro'
# }


# rnd_search_xgb = RandomizedSearchCV(
#     full_pipeline_xgb, param_distributions=param_distrbs, n_iter=10, cv=10,
#     scoring=scoring, refit='roc_auc', random_state=42, n_jobs=-1
# )

# rnd_search_xgb.fit(df_obesity, df_obesity_labels_2)

# best_params_xgb = rnd_search_xgb.best_params_
# print("Best Parameters of XGBooster:")
# print(best_params_xgb)


# best_scores_xgb = rnd_search_xgb.best_score_
# print("Best Scores of XGBooster:")
# print(best_scores_xgb)

<!--  -->

In [ ]:
# @title KNN Classifier (Hyperparameter Tuning) (Multiple parameters)

# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, f1_score




# # Create a pipeline
# full_pipeline_KNN = Pipeline([
#     ('preprocessing', preprocessing),
#     ('KNN_class', KNeighborsClassifier())
# ])

# # Update the parameter distribution dictionary with multiple parameters
# k_range = list(range(1, 31))
# weight_options = ['uniform', 'distance']

# params_KNN = dict(KNN_class__n_neighbors=k_range, KNN_class__weights=weight_options)

# # Define multiple scoring metrics
# scoring = {
#     'accuracy': 'accuracy',
#     'precision': 'precision_macro',
#     'roc_auc': 'roc_auc_ovr',
#     'f1': 'f1_macro'
# }

# # Create RandomizedSearchCV with multiple scoring
# rnd_search_KNN = RandomizedSearchCV(
#     full_pipeline_KNN, param_distributions=params_KNN, n_iter=10, cv=10,
#     scoring=scoring, refit='roc_auc', random_state=42, n_jobs=-1
# )

# # Fit the RandomizedSearchCV
# rnd_search_KNN.fit(df_obesity, df_obesity_labels_2)

# # Get the best parameters for each scoring metric
# best_params_KNN = rnd_search_KNN.best_params_
# print("Best Parameters:")
# print(best_params_KNN)

# # Get the best scores for each scoring metric
# best_scores_KNN = rnd_search_KNN.best_score_
# print("Best Scores:")
# print(best_scores_KNN)

# Testing on the test set

In [ ]:
# @title DT classifier

X_test_1 = strat_test_set.drop('BMXBMI',axis=1)
X_test = X_test_1.drop('BMI_cat',axis=1)

y_test_1 = strat_test_set['BMXBMI'].copy()
y_test_2 = strat_test_set['BMI_cat'].copy()

# # # print(np.unique(y_test_2))

# predictions_DT_class = tree_class.predict(X_test)
# final_acc_DT_class = accuracy_score(y_test_2, predictions_DT_class)
# print(final_acc_DT_class)



In [ ]:
# @title RF classifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
X_test_1 = strat_test_set.drop('BMXBMI',axis=1)
X_test = X_test_1.drop('BMI_cat',axis=1)

y_test_1 = strat_test_set['BMXBMI'].copy()
y_test_2 = strat_test_set['BMI_cat'].copy()

predictions_RF_class = forest_class.predict(X_test)
final_acc_RF_class = accuracy_score(y_test_2, predictions_RF_class)
print(final_acc_RF_class)


In [ ]:
# @title SVC classifier

# predictions_SVC_class = SVC_class.predict(X_test)
# final_acc_SVC_class = accuracy_score(y_test_2, predictions_SVC_class)
# print(final_acc_SVC_class)

In [ ]:
# @title XGBooster

# X_test_1 = strat_test_set.drop('BMXBMI',axis=1)
# X_test = X_test_1.drop('BMI_cat',axis=1)

# y_test_1 = strat_test_set['BMXBMI'].copy()
# y_test_2 = strat_test_set['BMI_cat'].copy()

# predictions_XGB_class = XGB_class.predict(X_test)
# final_acc_XGB_class =accuracy_score(y_test_2, predictions_XGB_class)
# print(final_acc_XGB_class)

In [ ]:
# @title KNN classifier

# X_test_1 = strat_test_set.drop('BMXBMI',axis=1)
# X_test = X_test_1.drop('BMI_cat',axis=1)

# y_test_1 = strat_test_set['BMXBMI'].copy()
# y_test_2 = strat_test_set['BMI_cat'].copy()

# predictions_KNN_class = KNN_class.predict(X_test)
# final_acc_KNN_class =accuracy_score(y_test_2, predictions_KNN_class)
# print(final_acc_KNN_class)

# Assessing the performance of the selected models

In [ ]:
# @title DT classifier (Complete performance metrics)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix



accuracy_DT_class = accuracy_score(y_test_2, predictions_DT_class)
print(accuracy_DT_class)

precision_DT_class = precision_score(y_test_2, predictions_DT_class)
print(precision_DT_class)

recall_DT_class = recall_score(y_test_2, predictions_DT_class)
print(recall_DT_class)

F1_DT_class = f1_score(y_test_2, predictions_DT_class)
print(F1_DT_class)

roc_auc_DT_class = roc_auc_score(y_test_2, tree_class.predict_proba(X_test)[:, 1])
print(roc_auc_DT_class)


## Visualization of ROC
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


probs = tree_class.predict_proba(X_test)
preds = probs[:,1]
fpr, tpr, threshold = roc_curve(y_test_2, preds)
roc_auc = auc(fpr, tpr)

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()


In [ ]:
# @title RF classifier (Complete performance metrics)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy_RF_class = accuracy_score(y_test_2, predictions_RF_class)
print(accuracy_RF_class)

precision_RF_class = precision_score(y_test_2, predictions_RF_class)
print(precision_RF_class)

recall_RF_class = recall_score(y_test_2, predictions_RF_class)
print(recall_RF_class)

F1_RF_class = f1_score(y_test_2, predictions_RF_class)
print(F1_RF_class)

roc_auc_RF_class = roc_auc_score(y_test_2, forest_class.predict_proba(X_test)[:, 1])
print(roc_auc_RF_class)


## Visualization of ROC
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


probs = forest_class.predict_proba(X_test)
preds = probs[:,1]
fpr, tpr, threshold = roc_curve(y_test_2, preds)
roc_auc = auc(fpr, tpr)

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()


In [ ]:
# @title SVC classifier (Complete performance metrics)


accuracy_SVC_class = accuracy_score(y_test_2, predictions_SVC_class)
print(accuracy_SVC_class)

precision_SVC_class = precision_score(y_test_2, predictions_SVC_class)
print(precision_SVC_class)

recall_SVC_class = recall_score(y_test_2, predictions_SVC_class)
print(recall_SVC_class)

F1_SVC_class = f1_score(y_test_2, predictions_SVC_class)
print(F1_SVC_class)

roc_auc_SVC_class = roc_auc_score(y_test_2, SVC_class.predict_proba(X_test)[:, 1])
print(roc_auc_SVC_class)

## Visualization of ROC
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


probs = SVC_class.predict_proba(X_test)
preds = probs[:,1]
fpr, tpr, threshold = roc_curve(y_test_2, preds)
roc_auc = auc(fpr, tpr)

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel



In [ ]:
# @title XGBoost Classifier (Complete performance metrics)

from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


accuracy_XGB_class = accuracy_score(y_test_2, predictions_XGB_class)
print(accuracy_XGB_class)

precision_XGB_class = precision_score(y_test_2, predictions_XGB_class)
print(precision_XGB_class)

recall_XGB_class = recall_score(y_test_2, predictions_XGB_class)
print(recall_XGB_class)

F1_XGB_class = f1_score(y_test_2, predictions_XGB_class)
print(F1_XGB_class)

roc_auc_XGB_class = roc_auc_score(y_test_2, XGB_class.predict_proba(X_test)[:, 1])
print(roc_auc_XGB_class)


## Visualization of ROC
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


probs = XGB_class.predict_proba(X_test)
preds = probs[:,1]
fpr, tpr, threshold = roc_curve(y_test_2, preds)
roc_auc = auc(fpr, tpr)

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel


In [ ]:
# @title KNN Classifier (Complete performance metrics)

from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


accuracy_KNN_class = accuracy_score(y_test_2, predictions_KNN_class)
print(accuracy_KNN_class)

precision_KNN_class = precision_score(y_test_2, predictions_KNN_class)
print(precision_KNN_class)

recall_KNN_class = recall_score(y_test_2, predictions_KNN_class)
print(recall_KNN_class)

F1_KNN_class = f1_score(y_test_2, predictions_KNN_class)
print(F1_KNN_class)

roc_auc_KNN_class = roc_auc_score(y_test_2, KNN_class.predict_proba(X_test)[:, 1])
print(roc_auc_KNN_class)


## Visualization of ROC
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


probs = KNN_class.predict_proba(X_test)
preds = probs[:,1]
fpr, tpr, threshold = roc_curve(y_test_2, preds)
roc_auc = auc(fpr, tpr)

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel


In [ ]:
#@title Learning Curve (Cross-entropy (log) Loss) (RF classifier)

from sklearn.metrics import log_loss
X_train = df_obesity
y_train = df_obesity_labels_2

X_test = X_test
y_test = y_test_2


n_trees = np.arange(1, 101)
rf = RandomForestClassifier(
    n_estimators=1,
    warm_start=True,
    max_depth=61,
    max_features=24,
    min_samples_split=81,
    random_state=42
)

# Create the full pipeline
model_pipeline = Pipeline([
    ("preprocessor", preprocessing),
    ("randomforestclassifier", rf)
])

# Arrays to store loss values
train_losses = []
test_losses = []
n_trees = np.arange(1, 101)  # Track trees from 1 to 100

# Train incrementally and calculate loss
for n in n_trees:
    model_pipeline.named_steps["randomforestclassifier"].set_params(n_estimators=n)

    model_pipeline.fit(X_train, y_train)

    train_probs = model_pipeline.predict_proba(X_train)
    test_probs = model_pipeline.predict_proba(X_test)

    # Compute log loss (cross-entropy)
    train_loss = log_loss(y_train, train_probs)
    test_loss = log_loss(y_test, test_probs)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

# Plot the loss curve
plt.figure(figsize=(8, 5))
plt.plot(n_trees, train_losses, label="Training Loss", color='blue')
plt.plot(n_trees, test_losses, label="Test Loss", color='red')
plt.xlabel("Number of Trees")
plt.ylabel("Cross-Entropy Loss (Log Loss)")
plt.title("Learning Curve: Training vs Test Loss")
plt.legend()
plt.grid(True)
plt.show()

# Feature Importance

In [ ]:
# @title SHAP (SHapley Additive exPlanations) (Random forest)

!pip install shap
from sklearn.ensemble import RandomForestClassifier

selector_SHAP = RandomForestClassifier(random_state=42, max_depth=61, max_features=24, min_samples_split=81)

from sklearn.ensemble import RandomForestClassifier



preprocessing = make_column_transformer(
    (num_pipeline, make_column_selector(dtype_include=np.number)),
    (cat_pipeline, make_column_selector(dtype_include='category'))
)

pipeline_selector_SHAP = Pipeline([
    ('preprocessing', preprocessing),
    ('feature_selector', selector_SHAP)
])

selected_features_SHAP = pipeline_selector_SHAP.fit(df_obesity,df_obesity_labels_2)

model_for_shap = pipeline_selector_SHAP.named_steps['feature_selector']

import shap

# Calculate SHAP values
preprocessed_df_obesity = pipeline_selector_SHAP.named_steps['preprocessing'].transform(df_obesity)
# Get feature names after preprocessing
transformed_column_names = pipeline_selector_SHAP.named_steps['preprocessing'].get_feature_names_out()



# Calculate SHAP values on the preprocessed data
explainer = shap.Explainer(model_for_shap, preprocessed_df_obesity)
shap_values = explainer.shap_values(preprocessed_df_obesity)




# Calculate mean absolute SHAP values for each feature
mean_abs_shap_values = np.mean([np.abs(sv) for sv in shap_values], axis=0)



mean_abs_shap_values_list = mean_abs_shap_values.tolist()


# Combine feature names and their corresponding mean SHAP values
feature_importance = list(zip(transformed_column_names, mean_abs_shap_values_list))

# Sort features based on mean absolute SHAP values (importance)
feature_importance_sorted = sorted(feature_importance, key=lambda x: x[1], reverse=True)

# Display the sorted feature importances
for feature, importance in feature_importance_sorted:
    print(f"{feature}: {importance}")

# Domain importances

In [ ]:
# @title Introduction of Domains

feature_to_domain = {
'pipeline-1__RIDAGEYR': 'Intrinsic factors',
'pipeline-1__INDFMPIR': 'Socioeconomic',
'pipeline-1__INDFMMPI': 'Socioeconomic',
'pipeline-1__PAD680': 'Physical activity',
'pipeline-1__SLD012': 'Sleep',
'pipeline-1__SLD013': 'Sleep',
'pipeline-1__DR1T_A_DRINKS': 'Dietary' ,
'pipeline-1__Depression_total' : 'Depression',
'pipeline-1__Total Fruits': 'Dietary' ,
'pipeline-1__Whole Fruits': 'Dietary' ,
'pipeline-1__Total Vegetables' : 'Dietary' ,
'pipeline-1__Greens and Beans': 'Dietary' ,
'pipeline-1__Whole Grains': 'Dietary' ,
'pipeline-1__Dairy': 'Dietary' ,
'pipeline-1__Total Protein Foods': 'Dietary' ,
'pipeline-1__Seafood and Plant Proteins': 'Dietary' ,
'pipeline-1__Fatty Acids Ratio (PUFAs + MUFAs)/SFAs': 'Dietary' ,
'pipeline-1__Refined Grains': 'Dietary' ,
'pipeline-1__Sodium' :'Dietary' ,
'pipeline-1__Added Sugars': 'Dietary' ,
'pipeline-1__Saturated Fats':'Dietary' ,
'pipeline-1__HEI-2020': 'Dietary' ,
'pipeline-1__Calorie_intake_mean':'Dietary' ,
'pipeline-1__Protein_intake_mean': 'Dietary' ,
'pipeline-1__Carb_intake_mean': 'Dietary' ,
'pipeline-1__Fat_intake_mean': 'Dietary' ,
'pipeline-1__Sugar_intake_mean': 'Dietary' ,
'pipeline-1__Fiber_intake_mean': 'Dietary' ,
'pipeline-1__SAFA_intake_mean': 'Dietary' ,
'pipeline-1__MUFA_intake_mean': 'Dietary' ,
'pipeline-1__PUFA_intake_mean': 'Dietary' ,
'pipeline-1__Cholesterol_intake_mean': 'Dietary' ,
'pipeline-2__RIAGENDR_1.0': 'Intrinsic factors',
'pipeline-2__RIAGENDR_2.0': 'Intrinsic factors',
'pipeline-2__RIDRETH3_1.0': 'Intrinsic factors',
'pipeline-2__RIDRETH3_2.0': 'Intrinsic factors',
'pipeline-2__RIDRETH3_3.0': 'Intrinsic factors',
'pipeline-2__RIDRETH3_4.0': 'Intrinsic factors',
'pipeline-2__RIDRETH3_6.0': 'Intrinsic factors',
'pipeline-2__RIDRETH3_7.0': 'Intrinsic factors',
'pipeline-2__DMDBORN4_1.0': 'Intrinsic factors',
'pipeline-2__DMDBORN4_2.0': 'Intrinsic factors',
'pipeline-2__DMDBORN4_77.0': 'Intrinsic factors',
'pipeline-2__DMDBORN4_99.0': 'Intrinsic factors',
'pipeline-2__DMDEDUC2_1.0': 'Intrinsic factors',
'pipeline-2__DMDEDUC2_2.0': 'Intrinsic factors',
'pipeline-2__DMDEDUC2_3.0': 'Intrinsic factors',
'pipeline-2__DMDEDUC2_4.0': 'Intrinsic factors',
'pipeline-2__DMDEDUC2_5.0': 'Intrinsic factors',
'pipeline-2__DMDEDUC2_9.0': 'Intrinsic factors',
'pipeline-2__DMDMARTZ_1.0': 'Intrinsic factors',
'pipeline-2__DMDMARTZ_2.0': 'Intrinsic factors',
'pipeline-2__DMDMARTZ_3.0': 'Intrinsic factors',
'pipeline-2__DMDMARTZ_77.0': 'Intrinsic factors',
'pipeline-2__CBQ506_1.0': 'Behavioral',
'pipeline-2__CBQ506_2.0': 'Behavioral',
'pipeline-2__CBQ536_1.0': 'Behavioral',
'pipeline-2__CBQ536_2.0': 'Behavioral',
'pipeline-2__CBQ536_9.0': 'Behavioral',
'pipeline-2__CBQ551_1.0': 'Behavioral',
'pipeline-2__CBQ551_2.0': 'Behavioral',
'pipeline-2__CBQ830_1.0': 'Behavioral',
'pipeline-2__CBQ830_2.0': 'Behavioral',
'pipeline-2__CBQ845_1.0': 'Behavioral',
'pipeline-2__CBQ845_2.0': 'Behavioral',
'pipeline-2__CBQ860_1.0': 'Behavioral',
'pipeline-2__CBQ860_2.0': 'Behavioral',
'pipeline-2__CBQ875_1.0': 'Behavioral',
'pipeline-2__CBQ875_2.0': 'Behavioral',
'pipeline-2__CBQ875_9.0': 'Behavioral',
'pipeline-2__CBQ890_1.0': 'Behavioral',
'pipeline-2__CBQ890_2.0': 'Behavioral',
'pipeline-2__CBQ890_9.0': 'Behavioral',
'pipeline-2__CBQ645_1.0': 'Behavioral',
'pipeline-2__CBQ645_2.0': 'Behavioral',
'pipeline-2__CBQ645_3.0': 'Behavioral',
'pipeline-2__CBQ645_4.0': 'Behavioral',
'pipeline-2__CBQ645_5.0': 'Behavioral',
'pipeline-2__CBQ645_6.0': 'Behavioral',
'pipeline-2__CBQ645_7.0': 'Behavioral',
'pipeline-2__CBQ645_99.0': 'Behavioral',
'pipeline-2__CBQ700_1.0': 'Behavioral',
'pipeline-2__CBQ700_2.0': 'Behavioral',
'pipeline-2__CBQ700_3.0': 'Behavioral',
'pipeline-2__CBQ700_4.0': 'Behavioral',
'pipeline-2__CBQ700_5.0': 'Behavioral',
'pipeline-2__CBQ700_6.0': 'Behavioral',
'pipeline-2__CBQ700_9.0': 'Behavioral',
'pipeline-2__DBQ780_1.0': 'Behavioral',
'pipeline-2__DBQ780_2.0': 'Behavioral',
'pipeline-2__DBQ780_3.0': 'Behavioral',
'pipeline-2__DBQ780_4.0': 'Behavioral',
'pipeline-2__DBQ780_5.0': 'Behavioral',
'pipeline-2__DBQ780_6.0': 'Behavioral',
'pipeline-2__DBQ780_9.0': 'Behavioral',
'pipeline-2__DBQ750_1.0': 'Behavioral',
'pipeline-2__DBQ750_2.0': 'Behavioral',
'pipeline-2__DBQ750_3.0': 'Behavioral',
'pipeline-2__DBQ750_4.0': 'Behavioral',
'pipeline-2__DBQ750_5.0': 'Behavioral',
'pipeline-2__DBQ750_9.0': 'Behavioral',
'pipeline-2__DBQ760_1.0': 'Behavioral',
'pipeline-2__DBQ760_2.0': 'Behavioral',
'pipeline-2__DBQ760_3.0': 'Behavioral',
'pipeline-2__DBQ760_4.0': 'Behavioral',
'pipeline-2__DBQ760_5.0': 'Behavioral',
'pipeline-2__DBQ760_9.0': 'Behavioral',
'pipeline-2__DBQ770_1.0': 'Behavioral',
'pipeline-2__DBQ770_2.0': 'Behavioral',
'pipeline-2__DBQ770_3.0': 'Behavioral',
'pipeline-2__DBQ770_4.0': 'Behavioral',
'pipeline-2__DBQ770_5.0': 'Behavioral',
'pipeline-2__DBQ770_6.0': 'Behavioral',
'pipeline-2__DBQ770_9.0': 'Behavioral',
'pipeline-2__CBQ905_1.0': 'Behavioral',
'pipeline-2__CBQ905_2.0': 'Behavioral',
'pipeline-2__CBQ905_3.0': 'Behavioral',
'pipeline-2__CBQ905_4.0': 'Behavioral',
'pipeline-2__CBQ905_5.0': 'Behavioral',
'pipeline-2__CBQ905_6.0': 'Behavioral',
'pipeline-2__CBQ905_9.0': 'Behavioral',
'pipeline-2__CBQ910_1.0': 'Behavioral',
'pipeline-2__CBQ910_2.0': 'Behavioral',
'pipeline-2__CBQ910_3.0': 'Behavioral',
'pipeline-2__CBQ910_4.0': 'Behavioral',
'pipeline-2__CBQ910_5.0': 'Behavioral',
'pipeline-2__CBQ910_6.0': 'Behavioral',
'pipeline-2__CBQ910_9.0': 'Behavioral',
'pipeline-2__CBQ685_1.0': 'Behavioral',
'pipeline-2__CBQ685_2.0': 'Behavioral',
'pipeline-2__CBQ685_3.0': 'Behavioral',
'pipeline-2__CBQ685_4.0': 'Behavioral',
'pipeline-2__CBQ685_5.0': 'Behavioral',
'pipeline-2__CBQ685_6.0': 'Behavioral',
'pipeline-2__CBQ685_9.0': 'Behavioral',
'pipeline-2__CBQ915_1.0': 'Behavioral',
'pipeline-2__CBQ915_2.0': 'Behavioral',
'pipeline-2__CBQ915_3.0': 'Behavioral',
'pipeline-2__CBQ915_4.0': 'Behavioral',
'pipeline-2__CBQ915_5.0': 'Behavioral',
'pipeline-2__CBQ915_6.0': 'Behavioral',
'pipeline-2__CBQ915_9.0': 'Behavioral',
'pipeline-2__CBD925_1.0': 'Behavioral',
'pipeline-2__CBD925_2.0': 'Behavioral',
'pipeline-2__CBD925_3.0': 'Behavioral',
'pipeline-2__CBD925_7.0': 'Behavioral',
'pipeline-2__CBD925_9.0': 'Behavioral',
'pipeline-2__CBQ930_1.0': 'Behavioral',
'pipeline-2__CBQ930_2.0': 'Behavioral',
'pipeline-2__CBQ930_3.0': 'Behavioral',
'pipeline-2__CBQ930_4.0': 'Behavioral',
'pipeline-2__CBQ930_5.0': 'Behavioral',
'pipeline-2__CBQ930_6.0': 'Behavioral',
'pipeline-2__CBQ930_9.0': 'Behavioral',
'pipeline-2__CBQ935_1.0': 'Behavioral',
'pipeline-2__CBQ935_2.0': 'Behavioral',
'pipeline-2__CBQ935_3.0': 'Behavioral',
'pipeline-2__CBQ935_4.0': 'Behavioral',
'pipeline-2__CBQ935_5.0': 'Behavioral',
'pipeline-2__CBQ935_6.0': 'Behavioral',
'pipeline-2__CBQ935_9.0': 'Behavioral',
'pipeline-2__CBQ945_1.0': 'Behavioral',
'pipeline-2__CBQ945_2.0': 'Behavioral',
'pipeline-2__CBQ945_3.0': 'Behavioral',
'pipeline-2__CBQ945_4.0': 'Behavioral',
'pipeline-2__CBQ945_5.0': 'Behavioral',
'pipeline-2__CBQ945_6.0': 'Behavioral',
'pipeline-2__CBQ945_9.0': 'Behavioral',
'pipeline-2__CBQ950_1.0': 'Behavioral',
'pipeline-2__CBQ950_2.0': 'Behavioral',
'pipeline-2__CBQ950_3.0': 'Behavioral',
'pipeline-2__CBQ950_4.0': 'Behavioral',
'pipeline-2__CBQ950_5.0': 'Behavioral',
'pipeline-2__CBQ950_6.0': 'Behavioral',
'pipeline-2__CBQ950_9.0': 'Behavioral',
'pipeline-2__DBQ700_1.0': 'Behavioral',
'pipeline-2__DBQ700_2.0': 'Behavioral',
'pipeline-2__DBQ700_3.0': 'Behavioral',
'pipeline-2__DBQ700_4.0': 'Behavioral',
'pipeline-2__DBQ700_5.0': 'Behavioral',
'pipeline-2__DBQ700_9.0': 'Behavioral',
'pipeline-2__CBQ596_1.0': 'Behavioral',
'pipeline-2__CBQ596_2.0': 'Behavioral',
'pipeline-2__CBQ596_9.0': 'Behavioral',
'pipeline-2__DRQSDIET_1.0': 'Behavioral',
'pipeline-2__DRQSDIET_2.0': 'Behavioral',
'pipeline-2__DR2STY_1.0': 'Behavioral',
'pipeline-2__DR2STY_2.0': 'Behavioral',
'pipeline-2__FSDHH_1.0': 'Socioeconomic',
'pipeline-2__FSDHH_2.0': 'Socioeconomic',
'pipeline-2__FSDHH_3.0': 'Socioeconomic',
'pipeline-2__FSDHH_4.0': 'Socioeconomic',
'pipeline-2__FSDAD_1.0': 'Socioeconomic',
'pipeline-2__FSDAD_2.0': 'Socioeconomic',
'pipeline-2__FSDAD_3.0': 'Socioeconomic',
'pipeline-2__FSDAD_4.0': 'Socioeconomic',
'pipeline-2__INDFMMPC_1.0': 'Socioeconomic',
'pipeline-2__INDFMMPC_2.0': 'Socioeconomic',
'pipeline-2__INDFMMPC_3.0': 'Socioeconomic',
'pipeline-2__INDFMMPC_7.0': 'Socioeconomic',
'pipeline-2__INDFMMPC_9.0': 'Socioeconomic',
'pipeline-2__MCQ300C_1.0': 'Intrinsic factors',
'pipeline-2__MCQ300C_2.0': 'Intrinsic factors',
'pipeline-2__MCQ300C_9.0': 'Intrinsic factors',
'pipeline-2__MCQ300A_1.0': 'Intrinsic factors',
'pipeline-2__MCQ300A_2.0': 'Intrinsic factors',
'pipeline-2__MCQ300A_9.0': 'Intrinsic factors',
'pipeline-2__PAQ605_1.0': 'Physical activity',
'pipeline-2__PAQ605_2.0': 'Physical activity',
'pipeline-2__PAQ605_9.0': 'Physical activity',
'pipeline-2__PAQ620_1.0': 'Physical activity',
'pipeline-2__PAQ620_2.0': 'Physical activity',
'pipeline-2__PAQ620_9.0': 'Physical activity',
'pipeline-2__PAQ635_1.0': 'Physical activity',
'pipeline-2__PAQ635_2.0': 'Physical activity',
'pipeline-2__PAQ635_9.0': 'Physical activity',
'pipeline-2__PAQ650_1.0': 'Physical activity',
'pipeline-2__PAQ650_2.0': 'Physical activity',
'pipeline-2__PAQ665_1.0': 'Physical activity',
'pipeline-2__PAQ665_2.0': 'Physical activity',
'pipeline-2__PAQ665_9.0': 'Physical activity',
'pipeline-2__SLQ030_0.0': 'Sleep',
'pipeline-2__SLQ030_1.0': 'Sleep',
'pipeline-2__SLQ030_2.0': 'Sleep',
'pipeline-2__SLQ030_3.0': 'Sleep',
'pipeline-2__SLQ030_7.0': 'Sleep',
'pipeline-2__SLQ030_9.0': 'Sleep',
'pipeline-2__SLQ040_0.0': 'Sleep',
'pipeline-2__SLQ040_1.0': 'Sleep',
'pipeline-2__SLQ040_2.0': 'Sleep',
'pipeline-2__SLQ040_3.0': 'Sleep',
'pipeline-2__SLQ040_7.0': 'Sleep',
'pipeline-2__SLQ040_9.0': 'Sleep',
'pipeline-2__SLQ050_1.0': 'Sleep',
'pipeline-2__SLQ050_2.0': 'Sleep',
'pipeline-2__SLQ050_9.0': 'Sleep',
'pipeline-2__SLQ120_0.0': 'Sleep',
'pipeline-2__SLQ120_1.0': 'Sleep',
'pipeline-2__SLQ120_2.0': 'Sleep',
'pipeline-2__SLQ120_3.0': 'Sleep',
'pipeline-2__SLQ120_4.0': 'Sleep',
'pipeline-2__SLQ120_9.0': 'Sleep'
}



In [ ]:
# @title Domain importances

!pip install shap
from sklearn.ensemble import RandomForestClassifier
import shap

domain_names = set(feature_to_domain.values())

# Create a dictionary to map features to each domain
domain_features = {domain: [] for domain in domain_names}
for feature, domain in feature_to_domain.items():
    domain_features[domain].append(feature)


selector_SHAP = RandomForestClassifier(random_state=42, max_depth=61, max_features=24, min_samples_split=81)
pipeline_selector_SHAP = Pipeline([
    ('preprocessing', preprocessing),
    ('feature_selector', selector_SHAP)
])

# Fit the pipeline to the training data
pipeline_selector_SHAP.fit(df_obesity, df_obesity_labels_2)


transformed_df = pipeline_selector_SHAP.named_steps['preprocessing'].transform(df_obesity)


transformed_column_names = pipeline_selector_SHAP.named_steps['preprocessing'].get_feature_names_out()


X_domain = {}

for domain, features in domain_features.items():
    domain_cols_mask = np.isin(transformed_column_names, features)
    X_domain[domain] = transformed_df[:, domain_cols_mask]




y = df_obesity_labels_2

domain_importances = {}


for domain, X_train_domain in X_domain.items():
    pipeline_selector_SHAP.named_steps['feature_selector'].fit(X_train_domain, y)
    explainer = shap.Explainer(pipeline_selector_SHAP.named_steps['feature_selector'], X_train_domain)
    shap_values = explainer(X_train_domain)

    # Calculate the mean absolute SHAP values
    domain_importances[domain] = np.mean(np.abs(shap_values.values))

sorted_domain_importances= {k: v for k, v in sorted(domain_importances.items(), key=lambda item: item[1], reverse=True)}

print("\nDomain Importances (Training Set):")
for domain, importance in sorted_domain_importances.items():
    print(f"{domain}: {importance:.4f}")


# LSTM (Simulation)

In [ ]:
#@title Packages and Libraries

!pip install tensorflow scikit-learn
!pip install gradio
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
!pip install tensorflow-gpu


In [ ]:
#@title Long Short-term Memory (LTSM) (Simulation)

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt

X = df_final
y = df_obesity_labels_1
# Apply preprocessing pipeline to the dataset
X_preprocessed = preprocessing.fit_transform(X)

# Standardize the target variable
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

# Convert dataset into sequences for LSTM
def create_sequences(data_X, data_y, seq_length):
    X_seq, y_seq = [], []
    for i in range(len(data_X) - seq_length):  # Ensure index remains valid
        X_seq.append(data_X[i:i + seq_length])
        y_seq.append(data_y[i + seq_length] if i + seq_length < len(data_y) else data_y[-1])  # Prevent out-of-bounds error
    return np.array(X_seq), np.array(y_seq)

# Define sequence length (6 months)
seq_length = 6
X_seq, y_seq = create_sequences(X_preprocessed, y_scaled, seq_length)

# Split into train and test sets
split_idx = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

# Define LSTM model
model = Sequential([
    LSTM(64, activation='relu', return_sequences=True, input_shape=(seq_length, X_seq.shape[2])),
    Dropout(0.2),
    LSTM(32, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

# Compile model
model.compile(optimizer='adam', loss='mse')

# Train model
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

# Select a random individual from the dataset
random_idx = np.random.randint(len(X_test))
individual = X_test[random_idx]

# Simulate BMI predictions over 6 months
n_months = 6
bmi_predictions_without_intervention = []
bmi_predictions_with_intervention = []

for month in range(n_months):
    predicted_bmi_no_intervention = model.predict(individual.reshape(1, seq_length, -1)).flatten()[0]

    # Apply interventions
    modified_individual = individual.copy()
    modified_individual[:, 0] *= 0.90  # Reduce depression by 10%
    modified_individual[:, 1] *= 1.05  # Increase sleep duration by 5%
    modified_individual[:, 2] *= 0.90  # Reduce sedentary time by 10%

    predicted_bmi_with_intervention = model.predict(modified_individual.reshape(1, seq_length, -1)).flatten()[0]

    bmi_predictions_without_intervention.append(predicted_bmi_no_intervention)
    bmi_predictions_with_intervention.append(predicted_bmi_with_intervention)

    # Shift sequences
    individual = np.vstack((individual[1:], np.hstack((individual[-1, :-1], [predicted_bmi_no_intervention]))))
    modified_individual = np.vstack((modified_individual[1:], np.hstack((modified_individual[-1, :-1], [predicted_bmi_with_intervention]))))

# Convert BMI predictions back to original scale
bmi_predictions_without_intervention = scaler_y.inverse_transform(np.array(bmi_predictions_without_intervention).reshape(-1, 1)).flatten()
bmi_predictions_with_intervention = scaler_y.inverse_transform(np.array(bmi_predictions_with_intervention).reshape(-1, 1)).flatten()

# Plot results
plt.figure(figsize=(10, 5))
plt.plot(range(1, n_months + 1), bmi_predictions_without_intervention, label="Without Intervention", linestyle='--', marker='o', color='red')
plt.plot(range(1, n_months + 1), bmi_predictions_with_intervention, label="With Intervention", linestyle='-', marker='s', color='green')
plt.xlabel("Months")
plt.ylabel("Predicted BMI")
plt.title("Effect of Interventions on BMI Over Time")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#@title Long Short-term Memory (LTSM)


import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras import layers, models, initializers
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.losses import MeanSquaredError
import matplotlib.pyplot as plt
import joblib
import random
import os

# 🌱 Set a fixed seed for reproducibility
# Set random seeds
os.environ['PYTHONHASHSEED'] = '0'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Ensure deterministic operations in TensorFlow
os.environ['TF_DETERMINISTIC_OPS'] = '1'



# Load preprocessed data
X = df_final
y = df_obesity_labels_1

preprocessing = make_column_transformer(
    (num_pipeline, make_column_selector(dtype_include=np.number)),
    (cat_pipeline, make_column_selector(dtype_include='category'))
)

LTSM_model = Pipeline([
    ('preprocessing', preprocessing)
])

# Load or create a preprocessing pipeline

X_preprocessed = preprocessing.fit_transform(X)

# Standardize the target variable
y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1))

# Convert dataset into sequences for LSTM
def create_sequences(data_X, data_y, seq_length, initial_bmi):
    X_seq, y_seq = [], []
    for i in range(len(data_X) - seq_length + 1):  # Ensure valid index
        sequence_with_bmi = np.hstack((np.full((seq_length, 1), initial_bmi), data_X[i:i + seq_length]))
        X_seq.append(sequence_with_bmi)
        y_seq.append(data_y[min(i + seq_length, len(data_y) - 1)])  # Safe indexing

    return np.array(X_seq), np.array(y_seq)

# Define parameters
seq_length = 10  # Sequence length
initial_bmi = 27  # Example initial BMI

# Create sequences
X_seq, y_seq = create_sequences(X_preprocessed, y_scaled, seq_length, initial_bmi)

# Split into train and test sets
split_idx = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

# Define LSTM model

model = models.Sequential([
    layers.LSTM(64, kernel_initializer=initializers.glorot_uniform(seed=42), return_sequences=True),
    layers.LSTM(32, kernel_initializer=initializers.glorot_uniform(seed=42)),
    layers.Dense(1, kernel_initializer=initializers.glorot_uniform(seed=42))
])

# Compile model
model.compile(optimizer='adam', loss='mse')

# Train model
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

# Save trained model and scalers
model.save("bmi_lstm_model.h5")
joblib.dump(y_scaler, "y_scaler.pkl")

# 🎯 Load model and scalers for prediction
model = tf.keras.models.load_model("bmi_lstm_model.h5",
                                   custom_objects={'mse': MeanSquaredError()})  # Explicitly define MSE
y_scaler = joblib.load("y_scaler.pkl")

# Define individual characteristics
individual_characteristics = np.array([[8, 5, 6]], dtype=np.float64)  # Depression, Sleep, Sedentary

# 🚀 Create the input sequence (Initial BMI + Risk Factors)
individual_sequence_with_bmi = np.hstack((
    np.full((seq_length, 1), initial_bmi),
    np.tile(individual_characteristics, (seq_length, 1))
))

# 🛠️ Pad to match the required input size
padded_sequence = np.zeros((seq_length, X_seq.shape[2]))
padded_sequence[:, :individual_sequence_with_bmi.shape[1]] = individual_sequence_with_bmi

# 🔄 Predict BMI at the initial step
predicted_bmi = model.predict(padded_sequence.reshape(1, seq_length, -1)).flatten()[0]
predicted_bmi = y_scaler.inverse_transform([[predicted_bmi]])[0, 0]  # Convert back to original scale

# 🔄 Store predictions
bmi_predictions_without_intervention = [initial_bmi]
bmi_predictions_with_intervention = [initial_bmi]

# Simulate BMI predictions over time
n_months = 12  # Number of months
for _ in range(n_months - 1):
    predicted_bmi_no_intervention = model.predict(padded_sequence.reshape(1, seq_length, -1)).flatten()[0]
    predicted_bmi_no_intervention = y_scaler.inverse_transform([[predicted_bmi_no_intervention]])[0, 0]

    # 🛠️ Apply intervention
    modified_sequence = padded_sequence.copy()
    modified_sequence[:, 1] *= 0.90  # Reduce depression
    modified_sequence[:, 2] *= 1.05  # Increase sleep
    modified_sequence[:, 3] *= 0.90  # Reduce sedentary

    predicted_bmi_with_intervention = model.predict(modified_sequence.reshape(1, seq_length, -1)).flatten()[0]
    predicted_bmi_with_intervention = y_scaler.inverse_transform([[predicted_bmi_with_intervention]])[0, 0]

    # 📌 Store predictions
    bmi_predictions_without_intervention.append(predicted_bmi_no_intervention)
    bmi_predictions_with_intervention.append(predicted_bmi_with_intervention)

    # 🔄 Update sequences
    new_row_no_intervention = np.zeros((1, X_seq.shape[2]))
    new_row_no_intervention[0, 0] = predicted_bmi_no_intervention
    new_row_no_intervention[0, 1:4] = individual_characteristics.flatten()

    new_row_with_intervention = np.zeros((1, X_seq.shape[2]))
    new_row_with_intervention[0, 0] = predicted_bmi_with_intervention
    new_row_with_intervention[0, 1:4] = individual_characteristics.flatten()

    padded_sequence = np.vstack((padded_sequence[1:], new_row_no_intervention))
    modified_sequence = np.vstack((modified_sequence[1:], new_row_with_intervention))

# 📈 Plot results
plt.figure(figsize=(10, 6))
plt.plot(range(n_months), bmi_predictions_without_intervention, label="No Intervention", color="red", linestyle="--", marker="o")
plt.plot(range(n_months), bmi_predictions_with_intervention, label="With Intervention", color="green", linestyle="-", marker="o")
plt.xlabel("Months")
plt.ylabel("Predicted BMI")
plt.title("BMI Predictions with and without Intervention")
plt.legend()
plt.show()


In [ ]:
#@title BMI prediction UI


import gradio as gr
import numpy as np
import tensorflow as tf
import joblib
import matplotlib.pyplot as plt
from tensorflow.keras.losses import MeanSquaredError
import io as io

# Load trained model and scalers
custom_objects = {"mse": MeanSquaredError()}

model = tf.keras.models.load_model("bmi_lstm_model.h5", custom_objects={'mse': MeanSquaredError()})
y_scaler = joblib.load("y_scaler.pkl")

# Define parameters
seq_length = 10  # Sequence length
initial_bmi = 27  # Example initial BMI
n_months = 12  # Prediction duration
num_features = 253




# Prediction function
def predict_bmi(depression, sleep_hours, sedentary):
    individual_characteristics = np.array([[depression, sleep_hours, sedentary]], dtype=np.float32)
    individual_sequence_with_bmi = np.hstack((
        np.full((seq_length, 1), initial_bmi),
        np.tile(individual_characteristics, (seq_length, 1))
    ))
    padded_sequence = np.pad(individual_sequence_with_bmi, ((0, 0), (0, num_features - individual_sequence_with_bmi.shape[1])), mode='constant')
    padded_sequence[:, :individual_sequence_with_bmi.shape[1]] = individual_sequence_with_bmi

    bmi_predictions_without_intervention = [initial_bmi]
    bmi_predictions_with_intervention = [initial_bmi]

    for _ in range(n_months - 1):
        predicted_bmi_no_intervention = model.predict(padded_sequence.reshape(1, seq_length, -1)).flatten()[0]
        predicted_bmi_no_intervention = y_scaler.inverse_transform([[predicted_bmi_no_intervention]])[0, 0]

        modified_sequence = padded_sequence.copy()
        modified_sequence[:, 1] *= 0.90  # Reduce depression
        modified_sequence[:, 2] *= 1.05  # Increase sleep
        modified_sequence[:, 3] *= 0.90  # Reduce sedentary

        predicted_bmi_with_intervention = model.predict(modified_sequence.reshape(1, seq_length, -1)).flatten()[0]
        predicted_bmi_with_intervention = y_scaler.inverse_transform([[predicted_bmi_with_intervention]])[0, 0]

        bmi_predictions_without_intervention.append(predicted_bmi_no_intervention)
        bmi_predictions_with_intervention.append(predicted_bmi_with_intervention)

        new_row_no_intervention = np.zeros((1, padded_sequence.shape[1]))
        new_row_no_intervention[0, 0] = predicted_bmi_no_intervention
        new_row_no_intervention[0, 1:4] = individual_characteristics.flatten()

        new_row_with_intervention = np.zeros((1, modified_sequence.shape[1]))
        new_row_with_intervention[0, 0] = predicted_bmi_with_intervention
        new_row_with_intervention[0, 1:4] = individual_characteristics.flatten()

        padded_sequence = np.vstack((padded_sequence[1:], new_row_no_intervention))
        modified_sequence = np.vstack((modified_sequence[1:], new_row_with_intervention))

    # Plot results
    plt.figure(figsize=(8, 5))
    plt.plot(range(n_months), bmi_predictions_without_intervention, label="No Intervention", color="red", linestyle="--", marker="o")
    plt.plot(range(n_months), bmi_predictions_with_intervention, label="With Intervention", color="green", linestyle="-", marker="o")
    plt.xlabel("Months")
    plt.ylabel("Predicted BMI")
    plt.title("BMI Predictions with and without Intervention")
    plt.legend()
    plt.grid()
    plt.savefig("bmi_plot.png")  # Save for local use
    buf = io.BytesIO()
    plt.savefig(buf, format="png")  # Save to memory buffer
    buf.seek(0)

    return plt

    return "bmi_plot.png"

# Create Gradio UI
iface = gr.Interface(
    fn=predict_bmi,
    inputs=[
        gr.Slider(1, 10, step=1, label="Depression (Score)"),
        gr.Slider(4, 10, step=0.5, label="Sleep (Hours)"),
        gr.Slider(1, 10, step=1, label="Sedentary Behavior (Hours)"),
    ],
    outputs=gr.Plot(),
    title="BMI Prediction Model",
    description="Enter your depression, sleep, and sedentary behavior scores to predict your BMI trends with and without intervention."
)

# Launch UI
iface.launch(show_error = True)
